In [ ]:
import sys
sys.path.append('../../../src/')

import matplotlib.pyplot as plt

# Import methods
import QGP.Defocusing.Methods as Def
import QGP.Focusing.Methods as Foc
from QGP.Soliton_Gen import *
from Utilities import *
from Eigenvalues import *

import config

# Antidark

In [ ]:
# Create solitons
L = 50
N = 400_001
eps = 0.01
k = 0.5+eps
#0.1

pars = {"k":k}

c_1 = np.sqrt(2) + 0.1 #1 + np.sqrt(2)

# Theretical solitons
w_dark, x_dark = soliton(c_1, k, soliton_type="anti", L=L, N=N)#, as_func=True) #c=0

In [ ]:
h = x_dark[1] - x_dark[0]

D1 = D1_neumann_centered(N, h)
D2 = D2_neumann_centered(N, h)

M, A, B = build_linearized_matrix(w_dark, D1, D2, k, c_1)#, profs=(eta, theta))

evals, evecs = spla.eigs(M, k=128, which='LR')#, sigma=1)

In [ ]:
evals

In [ ]:
plt.plot(evals.real, evals.imag, ls="", ms=3)
plt.xlabel(r"$\mathrm{Re}(\lambda)$")
plt.ylabel(r"$\mathrm{Im}(\lambda)$", rotation="horizontal")
#plt.savefig(f"Antidark_eigs_c_{c_1:.4f}.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Tolerance for selecting nearly real eigenvalues
imag_tol = 1e-8

# Indices of eigenvalues with imaginary part close to zero
real_like_idx = np.where(np.abs(np.imag(evals)) < imag_tol)[0]

if real_like_idx.size == 0:
    raise RuntimeError("No eigenvalues found with imaginary part close to zero.")

# Among the nearly real eigenvalues, pick the one with largest real part
idx = real_like_idx[np.argmax(np.real(evals[real_like_idx]))]

lam = evals[idx]
vec = evecs[:, idx]

p = vec[:N]
q = vec[N:]

print("Largest non-complex eigenvalue =", lam)

In [ ]:
#plt.plot(x_dark, evecs[:, -2][:N].real, ms=0, label="p")
plt.plot(x_dark, evecs[:, -1][N:].real, ms=0, label="q")
plt.xlabel("x")

plt.xlim(-.5, .5)
#plt.ylabel("q")
plt.legend()
plt.show()


In [ ]:
Nt = 10_000
tf = 2.5e-4

w_comp = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_comp.set_init_array(w_dark)
w_comp.solve()

In [ ]:
w_max = np.abs(w_comp.w).max(axis=1)

In [ ]:
#j_max = 1628
plt.plot(w_comp.t, w_max, "x-", lw=.5, markeredgewidth=.5, ms=3)
plt.xlim(1e-4, 2.25e-4)
plt.ylim(1.68, 2.5)
plt.show()

In [ ]:
total_plot = len(w_comp.t[w_comp.t<0.00018])
indices = np.arange(0, total_plot, 100)
x_indices = np.where(np.abs(w_comp.x)<0.5)

for n in indices:
    plt.plot(w_comp.x[x_indices], norm(w_comp.w[n][x_indices]), "x-", lw=.5, markeredgewidth=.5, ms=5)

#plt.xlim(-0.02,0.02)
plt.ylim(2.8, 3)
plt.show()

In [ ]:
e = w_comp.get_energy()

In [ ]:
plt.plot(w_comp.t[indices], e[indices], "x-", lw=.5, markeredgewidth=.5, ms=5)

In [ ]:
norm_grad_w = w_comp.h * np_norm(np.gradient(w_comp.w, w_comp.h, axis=1), np.inf, axis=1)

In [ ]:
plt.plot(w_comp.t[indices], norm_grad_w[indices], "x-", lw=.5, markeredgewidth=.5, ms=5)
#plt.xlim(1.3e-4, 2.5e-4)
#plt.ylim(1.68, 2.5)

In [ ]:
norm_grad_w = np.sqrt(w_comp.h) * np_norm(np.gradient(w_comp.w, w_comp.h, axis=1), 2, axis=1)

In [ ]:
plt.plot(w_comp.t[indices], norm_grad_w[indices], "x-", lw=.5, markeredgewidth=.5, ms=5)
#plt.xlim(1.3e-4, 2.5e-4)
#plt.ylim(1.68, 2.5)

In [ ]:
w_comp.t[1]-w_comp.t[0]

In [ ]:
w_comp.dt

In [ ]:
tf/(w_comp.Nt-1)

In [ ]:
w_comp.t.shape

In [ ]:
w_comp.t[-1]